# Hands-On 3: Neighbourhoods and classification

Record a prediction before running each experiment.

In [ ]:
from pathlib import Path
import os
import sys
try:
    import mlcourse.setup
except ModuleNotFoundError:
    bases = [Path(os.environ.get("MLCOURSE_ROOT", Path.cwd())), Path.cwd(), Path("/content/pp-machine-learning")]
    for base in bases:
        for candidate in (base.resolve(), *base.resolve().parents):
            if (candidate / "src/mlcourse/setup.py").is_file():
                sys.path.insert(0, str(candidate / "src"))
                break
        else:
            continue
        break
    else:
        raise RuntimeError("Course files not found. Open the extracted course repository or set MLCOURSE_ROOT to its location.") from None

In [ ]:
from mlcourse.setup import setup_notebook
REPO_ROOT = setup_notebook()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from mlcourse.labs import load_course_data

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay
from mlcourse.labs import compare_boundaries, split_classification
from mlcourse.widgets import interactive_knn

## 1. Inspect Iris

Load the Iris dataset into a dataframe. Inspect its content with `head()` and `describe()`.

**Prediction:** Which columns can locate an observation in feature space?

*Your response.*

In [ ]:
iris = load_course_data('iris')
display(iris.head())
display(iris.describe())
display(iris.Species.value_counts().rename('count'))

**Observation:** How many measurements and classes are present?

*Your response.*

**Explanation:** Why should the species label be excluded from the predictors?

*Your response.*

## 2. Fit a neighbour classifier

Split the four measurements into training and test data. Fit scaled `KNeighborsClassifier(n_neighbors=5)`.

**Prediction:** How might unequal measurement scales change the nearest neighbours?

*Your response.*

In [ ]:
X = iris.drop(columns='Species')
y = iris.Species
X_train, X_test, y_train, y_test = split_classification(X, y)
knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
knn.fit(X_train, y_train)
print(f'Test accuracy: {accuracy_score(y_test, knn.predict(X_test)):.3f}')
fig, ax = plt.subplots(figsize=(7, 5), layout='constrained')
ConfusionMatrixDisplay.from_estimator(knn, X_test, y_test, ax=ax, colorbar=False, xticks_rotation=20)
plt.show()

**Observation:** Which off-diagonal entries contain errors?

*Your response.*

**Explanation:** Relate those errors to overlapping neighbourhoods.

*Your response.*

## 3. Compare two fixed boundaries

Use sepal measurements only. Compare `n_neighbors=1` with `n_neighbors=15` on the same split.

**Prediction:** Which boundary will be more sensitive to individual observations?

*Your response.*

In [ ]:
X2 = iris[['SepalLengthCm', 'SepalWidthCm']]
X2_train, X2_test, y2_train, y2_test = split_classification(X2, y)
comparison = compare_boundaries({
    'k=1': make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=1)),
    'k=15': make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15)),
}, X2_train, X2_test, y2_train, y2_test)
display(comparison)

**Observation:** Compare boundary detail and the two accuracy columns.

*Your response.*

**Explanation:** How does neighbourhood size change local sensitivity and smoothing?

*Your response.*

## 4. Manipulate the neighbourhood

Change `n_neighbors` from `1` to `31`, then compare uniform and distance weighting.

**Prediction:** At which end of the range do you expect the smoothest boundary?

*Your response.*

In [ ]:
knn_lab = interactive_knn(X2_train, X2_test, y2_train, y2_test)
display(knn_lab.widget)

**Observation:** Record one setting that changes a local region and its test accuracy.

*Your response.*

**Explanation:** Explain the change using neighbour votes and distance weighting.

*Your response.*

## 5. Compare inductive biases

Fit a linear discriminant model, Gaussian Naive Bayes and a shallow tree on `ds2`.

**Prediction:** Which model can form axis-aligned regions?

*Your response.*

In [ ]:
data = load_course_data('ds2')
a, b, c, d = split_classification(data[['X1', 'X2']], data.Class)
display(compare_boundaries({'Linear discriminant': LinearDiscriminantAnalysis(),
    'Gaussian Naive Bayes': GaussianNB(),
    'Tree': DecisionTreeClassifier(max_depth=3, random_state=0)}, a, b, c, d))

**Observation:** Compare the shapes of the three boundaries.

*Your response.*

**Explanation:** Which restrictions does each model place on the classification rule?

*Your response.*